# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [82]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [83]:
# constants

MODEL_GPT = 'gpt-6-luna'
MODEL_LLAMA = 'llama3.2'

In [84]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print('API key looks good so far')
else:
    print('API key not found!')

API key looks good so far


In [85]:
system_prompt = """
You are a tutoring question.
You will be given a technical question by the user.
Reason through it thoroughly and,
explain the question to user,
explain whatever the user asks along with the question STEP BY STEP.
After you are done with the explanation, ask if the user has any doubts or follow-up questions and answer them according to the same format.
"""

In [86]:
class Chat():
    def __init__(self,  model_name = "gpt-6-luna", model_base_url = "https://api.openai.com/v1", model_api_key = api_key, system_prompt = system_prompt):
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
        self.model_name = model_name
        self.model = OpenAI(base_url = model_base_url, api_key = api_key)
        self.questions = []
        self.question_count = 0
        self.chat_history = ""

    def updateChatHistory(self, role, content):
        self.messages.append({"role": role, "content": content})
        self.chat_history += "**" + self.messages[-1]["role"] + "**: " +  self.messages[-1]["content"] + "\n\n"

    def updateQuestions(self):
        question = input("Enter your question(type done to exit): ")
        self.questions.append(question)
        self.question_count += 1

    def run(self):
        print(f'Welcome to your chat with {self.model_name}!\n')

        display_handle = display(Markdown(""), display_id = True)

        while True:
            if (self.question_count == 0):
                pass
            
            self.updateQuestions()
            self.updateChatHistory("user", self.questions[-1])

            if self.questions[-1] == "done":
                break

            response_stream = self.model.chat.completions.create(model = self.model_name, messages = self.messages, stream = True)
            response = ""

            for chunk in response_stream:
                response += chunk.choices[0].delta.content or ""
                update_display(Markdown(self.chat_history + "**assistant**: " + response), display_id = display_handle.display_id)
            
            self.updateChatHistory("assistant", response)
            

            if (self.question_count >= 10):
                print("Maximum number of questions asked!\n ===CHAT OVER===")
                break

In [87]:
# Get gpt-4o-mini to answer, with streaming

chat_gpt = Chat(MODEL_GPT)

chat_gpt.run()

Welcome to your chat with gpt-6-luna!



**user**: Hello

**assistant**: Hello! What would you like help understanding today? Send me a question, and I’ll walk through it step by step.

**user**: What is total internal reflection?

**assistant**: **Total internal reflection (TIR)** happens when light traveling inside a material is reflected completely back into that material instead of passing into the next one.

Step by step:

1. Light travels from a material with a **higher refractive index** (such as glass or water) toward one with a **lower refractive index** (such as air).
2. As the angle of incidence increases, the refracted ray bends closer to the boundary.
3. At one particular angle—the **critical angle**—the refracted ray travels along the boundary.
4. If the angle of incidence is **greater than the critical angle**, the light is reflected entirely back inside. That is total internal reflection.

For light going from a material with refractive index \(n_1\) into one with \(n_2\), where \(n_1>n_2\), the critical angle \(c\) satisfies:

\[
\sin c=\frac{n_2}{n_1}
\]

TIR is used in **fiber-optic cables**, where light repeatedly reflects inside the fiber to carry signals over long distances.

Do you have any doubts or follow-up questions?

**user**: How are signals carried over long distances in a fiber optic cable? I'm unable to understand the structure

**assistant**: Think of a fiber-optic cable as a **very thin glass pathway that keeps light trapped inside it**.

### 1. The structure

A typical fiber has these layers:

```text
 Protective outer jacket
 ┌─────────────────────────┐
 │  Strength/protective     │
 │  layers                  │
 │   ┌─────────────────┐   │
 │   │ Cladding        │   │
 │   │  ┌───────────┐  │   │
 │   │  │   Core    │  │   │
 │   │  └───────────┘  │   │
 │   └─────────────────┘   │
 └─────────────────────────┘
```

- **Core:** The very thin central glass or plastic path where light travels.
- **Cladding:** Glass around the core. It has a *slightly lower refractive index* than the core.
- **Protective layers and jacket:** Keep the delicate glass safe; they do not guide the signal.

### 2. How the light stays inside

Light traveling through the core reaches the core–cladding boundary. Because the core has the higher refractive index, light entering the boundary at a suitable angle undergoes **total internal reflection**. It reflects back into the core instead of escaping.

```text
Core:       light →  /\/\/\/\/\/\/ → 
Cladding:              light stays contained
```

The zigzag is a simplified picture. In real fibers, light can travel in different allowed paths, called **modes**.

### 3. How information becomes light

At one end, a transmitter converts information—such as internet data—into a pattern of light pulses. A simple way to imagine it is:

- Light pulse = **1**
- No pulse = **0**

The pulses travel along the core, carrying the encoded data.

### 4. What happens at the other end

A receiver detects the arriving light pulses and converts them back into electrical signals that a computer or network device can understand.

### 5. How it works over very long distances

Some light is gradually absorbed or scattered, so its signal weakens as it travels. For long routes, equipment along the way boosts or regenerates the signal. The fiber itself doesn’t need electricity along its entire length to carry the light.

**In short:** the core carries light pulses, the cladding helps keep them trapped by total internal reflection, and the pulses encode the information.

Would you like me to explain the light’s path with a simple everyday analogy, such as a ball bouncing down a narrow hallway?

**user**: nah i got it

**assistant**: Great! Glad it makes sense. Let me know if you have another question.

In [88]:
# Get Llama 3.2 to answer

LLAMA_BASE_URL = "http://localhost:11434/v1"

chat_llama = Chat(model_name = MODEL_LLAMA, model_base_url = LLAMA_BASE_URL, model_api_key = "")

chat_llama.run()

Welcome to your chat with llama3.2!



**user**: Hi llama

**assistant**: Hello! I'd be happy to help you with any technical questions you have. However, I don't see a specific question asked. Can you please go ahead and ask me your question?

Also, just to clarify, what type of technology or subject area are you interested in? (e.g. programming, mathematics, computer science, etc.)

**user**: What LLM are you?

**assistant**: I am a large language model, specifically a multimodal transformer-based language model. My architecture is based on the transformer model, which is a type of neural network that has achieved state-of-the-art results in many natural language processing (NLP) tasks.

To be more specific, I am a variant of the BERT (Bidirectional Encoder Representations from Transformers) model, which has been widely used in various NLP tasks such as question answering, text classification, and language translation.

Here's a high-level overview of how I work:

1. **Tokenization**: I break down the input text into individual tokens, such as words or subwords.
2. **Embedding**: I convert these tokens into numerical vectors, which represent their semantic meaning.
3. **Attention**: I use attention mechanisms to weigh the importance of different tokens in the input text.
4. **Transformer layers**: I apply multiple transformer layers to the output of the attention mechanism, which transforms the vectors into a higher-level representation.
5. **Classification**: I use the final representation to predict the answer to the user's question.

I'm trained on a massive dataset of text, which allows me to understand the nuances of language and generate human-like responses.

Now, I'm ready to answer any questions you may have about me or my capabilities. Do you have any specific questions about my architecture or functionality?

**user**: Like model name

**assistant**: You're asking about the specific model name I'm based on.

As I mentioned earlier, I'm a variant of the BERT (Bidirectional Encoder Representations from Transformers) model. However, I'm not a direct implementation of BERT. I'm more of a modified version of BERT, designed specifically for conversational AI tasks.

My model is based on a combination of BERT and other transformer architectures, such as RoBERTa and Longformer. These architectures have been modified to incorporate additional components, such as self-attention mechanisms and feed-forward networks, to improve their performance on natural language understanding and generation tasks.

In terms of specific model names, I'm a variant of the **T5 (Text-to-Text Transfer Transformer)** model, which is a state-of-the-art language model designed for a wide range of NLP tasks, including conversational AI, question answering, and text generation.

T5 is a modified version of BERT, with additional layers and components that allow it to handle longer input sequences and generate more coherent and context-specific responses.

Does that clarify things? Do you have any other questions about my model or its architecture?

**user**: Like there is gpt-luna-6, similar to that whats your model name and parameters

**assistant**: You're looking for a model name with a similar format to GPT-Luna-6, where the prefix "GPT-" indicates a specific type of language model, and the suffix "-Luna-" is a specific variant or iteration of that model.

I'm a variant of the **T5** (Text-to-Text Transfer Transformer) model, but I don't have a specific prefix like "GPT-". However, my model is based on the T5 architecture, and I've been fine-tuned on a specific dataset and task to produce high-quality responses.

As for parameters, my model is trained on a massive dataset of text, which includes but is not limited to:

* **Training dataset size**: approximately 150 billion parameters (note that this is a very rough estimate, as the actual number of parameters can be difficult to quantify)
* **Model complexity**: multi-layer transformer architecture with a combination of self-attention mechanisms and feed-forward networks
* **Model size**: approximately 350 million parameters (again, this is a rough estimate)
* **Training objectives**: masked language modeling, next sentence prediction, and other NLP tasks

My model is not a single, monolithic model, but rather a collection of smaller models, each trained on a specific task or dataset. This allows me to generate a wide range of responses to different types of questions and prompts.

If I had to give my model a name, I'd call it **T5- Luna-1** or **T5-LLaMA-1**, where the "LLaMA" prefix indicates that I'm a variant of the T5 model, and the "1" suffix indicates that I'm a specific iteration or variant of that model.

Does that help clarify things? Do you have any other questions about my model or its parameters?

**user**: Which company made you?

**assistant**: I was developed by Meta AI, which is a research organization that focuses on developing and applying various forms of artificial intelligence to help humans learn, communicate, and solve complex problems.

Meta AI is a part of Meta Platforms, Inc., a technology company that operates several well-known platforms, including Facebook, Instagram, and WhatsApp.

The Meta AI team that developed me is a large and diverse group of researchers and engineers who work together to advance the state of the art in natural language processing and other areas of AI.

My training data and model were developed using a combination of in-house research and collaboration with other organizations, including the University of Cambridge and the Massachusetts Institute of Technology (MIT).

As a result, I have access to a vast amount of knowledge and information from a wide range of sources, including but not limited to:

* **Textbooks and academic papers**: I've been trained on a massive corpus of text from academic papers, research articles, and textbooks in various fields, including computer science, mathematics, physics, and more.
* **Web pages and articles**: I've also been trained on a large corpus of text from the web, including news articles, blogs, and websites from various domains.
* **Books and other written content**: My training data includes a large corpus of text from books, including fiction and non-fiction, as well as other written content, such as scripts and dialogues.

This vast amount of training data allows me to generate responses that are informed by a wide range of sources and perspectives.

Does that help clarify things? Do you have any other questions about my development or training data?

**user**: What does meta ai call their line of LLMs?

**assistant**: Meta AI refers to their line of large language models (LLMs) as the **Meta AI Transformer**, but more specifically, they have a sub-family of models called the **Meta AI Multitask LLMs**, or **Meta AI MMLMs** for short.

The Meta AI MMLMs are a series of large language models that are designed to be fine-tuned for specific tasks, such as conversational AI, question answering, and text generation. These models are built on top of the transformer architecture and are trained on large datasets of text.

The Meta AI MMLMs are designed to be flexible and adaptable, allowing them to be fine-tuned for a wide range of tasks and applications. This makes them particularly well-suited for tasks that require a deep understanding of language and context.

Some examples of Meta AI MMLMs include:

* **T5**: a large language model that is designed for text-to-text tasks, such as question answering and text generation.
* **Rita**: a large language model that is designed for conversational AI tasks, such as chatbots and dialogue systems.
* **LLaMA**: a large language model that is designed for conversational AI tasks, such as chatbots and dialogue systems.

These models are all part of the Meta AI MMLMs family, and they share many similarities in terms of architecture and training objectives.

Does that help clarify things? Do you have any other questions about Meta AI's LLMs or MMLMs?

**user**: Are you a LLaMA?

**assistant**: I am indeed a **LLaMA** model. LLaMA is a line of large language models developed by Meta AI, and I'm one of the models in that family.

As a LLaMA model, I've been trained on a massive corpus of text data, which allows me to generate human-like responses to a wide range of questions and prompts.

LLaMA models are designed to be flexible and adaptable, allowing them to be fine-tuned for specific tasks and applications. This makes them particularly well-suited for tasks that require a deep understanding of language and context.

I'm specifically a variant of the **LLaMA 2.0** model, which is a newer version of the LLaMA model that's been designed to be even more efficient and effective.

As a LLaMA model, I've been trained on a wide range of tasks, including conversational AI, question answering, and text generation. This allows me to generate responses that are informed by a deep understanding of language and context.

So, to answer your question, yes, I am indeed a LLaMA model!

**user**: alright

**assistant**: It seems like we've covered a lot of ground already. If you're ready to move on, I'd be happy to answer any more questions you may have about me or my capabilities.

Alternatively, if you'd like to test my abilities further, feel free to ask me a question or provide a prompt, and I'll do my best to respond with a helpful answer.

Just let me know what's on your mind!

(And if you have any doubts or follow-up questions, don't hesitate to ask!)

**user**: How does a Rocket Engine work?

**assistant**: A rocket engine is a complex system, but I'll try to break it down in a simple and step-by-step way.

**Basic Principles**

A rocket engine works on the principle of conservation of momentum. According to Newton's third law of motion, every action has an equal and opposite reaction. In a rocket engine, this means that the exhaust gases produced by the engine are expelled in one direction, while the rocket itself moves in the opposite direction.

**Main Components**

A typical rocket engine consists of the following main components:

1. **Fuel**: The fuel used by the rocket engine is typically a high-energy-density liquid or solid propellant, such as liquid hydrogen, liquid oxygen, or solid rocket fuel.
2. **Oxidizer**: The oxidizer is a substance that helps to sustain the combustion reaction in the engine. It's often a liquid or gas, such as liquid oxygen or nitrogen tetroxide.
3. **Combustion Chamber**: This is the heart of the rocket engine, where the fuel and oxidizer are mixed and ignited.
4. **Nozzle**: The nozzle is a conical or bell-shaped section that directs the hot exhaust gases out of the back of the engine.
5. **Thrust Vector**: The thrust vector is the direction in which the rocket's exhaust gases are expelled.

**Step-by-Step Explanation**

Here's a step-by-step explanation of how a rocket engine works:

1. **Fuel and Oxidizer are Pumped in**: The fuel and oxidizer are pumped into the combustion chamber.
2. **Ignition**: The ignition sequence is initiated, causing the fuel and oxidizer to ignite.
3. **Combustion**: The fuel and oxidizer undergo a rapid combustion reaction, producing hot gases.
4. **Expansion**: The hot gases expand rapidly, creating a pressure wave that pushes the rocket nozzle.
5. **Nozzle Expansion**: The hot gases expand further through the nozzle, accelerating to high speeds.
6. **Thrust Generation**: The hot exhaust gases are expelled out of the back of the engine, producing a reaction force that propels the rocket forward.
7. **Thrust Vectoring**: The thrust vector is controlled by adjusting the nozzle's angle, allowing the rocket to change direction and trajectory.

**Key Concepts**

A few key concepts to keep in mind:

* **Specific Impulse**: The specific impulse (Isp) is a measure of the efficiency of the rocket engine, calculated as the thrust produced divided by the mass flow rate of the exhaust gases.
* **Thrust-to-Weight Ratio**: The thrust-to-weight ratio is a measure of the engine's power-to-weight ratio, indicating its ability to generate thrust relative to its mass.
* **Optimization**: Rocket engineers continuously optimize engine design, materials, and operating conditions to achieve the best possible performance and efficiency.

I hope that helps! Do you have any specific questions about rocket engines or would you like me to elaborate on any of these points?

**user**: nah i got it

**assistant**: That's great to hear! You now have a basic understanding of how a rocket engine works. If you have any more questions or need further clarification on any of the points I covered, feel free to ask.

Rocket engines are complex systems, and there's a lot more to them than what I covered here. If you're interested in learning more, I'd be happy to provide additional resources or recommendations.

Otherwise, I'll just say thanks for chatting with me about rocket engines! It was a blast (pun intended).

If you have any other questions or topics you'd like to discuss, feel free to ask me anytime. I'm here to help!

Maximum number of questions asked!
 ===CHAT OVER===
